In [1]:
# Standard libraries
import sys
from pathlib import Path
import time

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Third-party libraries
from gymnasium.utils.env_checker import check_env
import numpy as np
from envs.balance_bot_env import BalanceBotEnv


In [2]:
# Import the custom environment adapted for balancing_bot.xml
from env_setup import BalanceBotEnv

In [3]:
# Settings
SEED = 42

In [4]:
# Initialize environment; its default MJCF path is balancing_bot.xml beside the module
env = BalanceBotEnv(render_mode="human")

# Print observation and action spaces
print(env.observation_space)
print(env.action_space)

Box([ -3.1415927 -20.        -50.        -50.       ], [ 3.1415927 20.        50.        50.       ], (4,), float32)
Box(-1.0, 1.0, (2,), float32)


In [5]:
# Check the Gymnasium API without requiring a GUI window
check_env(env, skip_render_check=True)

In [6]:
# Reset environment and print observation
obs, info = env.reset(seed=SEED)
print(obs)

[0.00548753 0.08218682 0.         0.        ]


In [7]:
# Manual step with no action (don't turn the motors)
obs, reward, terminated, truncated, info = env.step(np.array([0.0, 0.0]))
print(f"obs: {obs}")
print(f"reward: {reward}")
print(f"terminated: {terminated}")
print(f"truncated: {truncated}")

obs: [0.01092019 0.08218682 0.         0.        ]
reward: 0.9983021743300764
terminated: False
truncated: False


In [8]:
# Test: random policy
obs, _ = env.reset()
total_reward = 0.0

# Go for 200 steps
terminated = False
for step in range(200):
    step_start = time.perf_counter()

    # Choose a random action
    action = env.action_space.sample()

    # Take a step with the given action
    obs, reward, terminated, truncated, _ = env.step(action)

    # Accumulate reward
    total_reward += reward

    # Render in MuJoCo
    env.render()

    # Stop if the episode ended
    if terminated or truncated:
        print(f"Episode ended at step {step}")
        break

    # Match wall-clock pacing to the model's configured timestep
    slack = env.model.opt.timestep - (time.perf_counter() - step_start)
    if slack > 0:
        time.sleep(slack)

print(f"Final obs: {obs}")
print(f"Final reward: {total_reward}")

Episode ended at step 37
Final obs: [ -0.539746  -9.986655 -12.533795  14.51243 ]
Final reward: -90.50214030106754


In [9]:
# Close the environment
env.close()